# CS55_1_demo

Cross-modal evidence-chain demo:

**Storyboard → Animatic → Final → CLIP → sequence-aware matching → metrics → libevchain Pipeline → JSON**

The packaged `src/libevchain/` is kept unchanged from the supplied COMP3988 source. CS55-1 code lives in `src/cs55_demo/`.

## 1. Choose dataset

Use `config.sample.json` for the packaged small test. For Sol Levante, copy and edit `config.sollevante.example.json`, then change the filename below.

In [ ]:
CONFIG_FILE = "config.sample.json"

## 2. Load the packaged CS55-1 system

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cs55_demo import load_config, run_full_pipeline, save_outputs

config = load_config(ROOT / CONFIG_FILE)
print("Dataset:", config["dataset_name"])

## 3. Run

The first real CLIP run may download `openai/clip-vit-base-patch32`. Later runs reuse the cache.

In [ ]:
cache_path = ROOT / "cache" / f"{config['dataset_name']}_embeddings.pt"

result, bundle = run_full_pipeline(
    config,
    use_cache=True,
    cache_path=cache_path,
)

print("Pipeline complete.")
print("Dataset:", result["dataset_name"])
print("Detailed results are displayed in the sections below.")

## 4. Content consistency results

These are the original cross-modal measurements from Storyboard → Animatic → Final. They are separate from the provenance evidence indicators below.

In [ ]:
import json
from html import escape
from IPython.display import HTML, Markdown, display


def _display_value(value, max_length=180):
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False, default=str)
    elif isinstance(value, float):
        text = f"{value:.4f}"
    else:
        text = str(value)
    return text if len(text) <= max_length else text[:max_length - 1] + "…"


def display_table(headers, rows):
    header_html = "".join(
        f"<th style='text-align:left;padding:8px;border-bottom:2px solid #999'>{escape(str(header))}</th>"
        for header in headers
    )
    body_html = "".join(
        "<tr>" + "".join(
            f"<td style='vertical-align:top;padding:8px;border-bottom:1px solid #ddd'>{escape(_display_value(value))}</td>"
            for value in row
        ) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto'>"
        "<table style='border-collapse:collapse;width:100%;font-size:14px'>"
        f"<thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody>"
        "</table></div>"
    ))


content_rows = [
    ("Chain coherence", result["chain_metrics"]["coherence"]),
    ("Confidence", result["chain_metrics"]["confidence"]),
    ("Evidence coverage", result["chain_metrics"]["coverage"]),
    ("Completeness", result["chain_metrics"]["completeness"]),
    (
        "Storyboard → Animatic similarity",
        result["stage_metrics"]["storyboard_to_animation_similarity"],
    ),
    (
        "Animatic → Final similarity",
        result["stage_metrics"]["animation_to_final_similarity"],
    ),
    ("EvidenceChain artefacts", result["evidence_chain"]["artefact_count"]),
    ("Evidence relationships", result["evidence_chain"]["relationship_count"]),
]

display_table(["Measurement", "Value"], content_rows)
display(Markdown(
    "*These measurements evaluate content consistency. They are not an authenticity probability.*"
))

## 5. Four provenance indicators — no score

This table reports whether evidence is available. `not_available` means the configured evidence was not verified or was not found; it does **not** mean the asset is fake.

In [ ]:
indicator_labels = [
    "File Integrity",
    "C2PA Provenance",
    "Source Project Evidence",
    "Editing / AI Transparency",
]

indicator_rows = []
for asset_name in ("storyboard", "animatic", "final"):
    statuses = result["provenance_authenticity_evidence"][asset_name]
    indicator_rows.append([
        asset_name.title(),
        *[
            "✓ available" if statuses[label] == "available" else "— not available"
            for label in indicator_labels
        ],
    ])

display_table(["Asset", *indicator_labels], indicator_rows)

## 6. Indicator inspection details

Storyboard, Animatic, and Final are all shown below. Every indicator includes its reason and the checks used to reach the availability status.

In [ ]:
for asset_name in ("storyboard", "animatic", "final"):
    asset_provenance = result["provenance"][asset_name]
    summary = asset_provenance["provenance_summary"]

    display(Markdown(
        f"### {asset_name.title()} — {summary['status']}  "
        f"\nAvailable indicators: **{summary['coverage']}** "
        "(availability count only; no score)"
    ))

    for indicator_id, indicator in summary["indicators"].items():
        status_symbol = "✓" if indicator["available"] else "—"
        display(Markdown(
            f"#### {status_symbol} {indicator['label']} — `{indicator['status']}`\n\n"
            f"{indicator['reason']}"
        ))
        check_rows = [
            (
                check["name"],
                "yes" if check["result"] else "no",
                check.get("value"),
                check["source"],
                check["explanation"],
            )
            for check in indicator["checks"]
        ]
        display_table(
            ["Check", "Result", "Value", "Source", "Explanation"],
            check_rows,
        )

    if summary["conflicts"]:
        display(Markdown("#### Conflicts requiring review"))
        display_table(
            ["Code", "Description", "Source"],
            [
                (item["code"], item["description"], item["source"])
                for item in summary["conflicts"]
            ],
        )

    if summary["warnings"]:
        display(Markdown("#### Warnings"))
        for warning in summary["warnings"]:
            display(Markdown(f"- {warning}"))

## 7. Detailed provenance evidence

The following views expose the underlying C2PA, software, project-file, and editing-history records for Storyboard, Animatic, and Final.

In [ ]:
for asset_name in ("storyboard", "animatic", "final"):
    asset_provenance = result["provenance"][asset_name]
    display(Markdown(f"## {asset_name.title()}"))

    c2pa = asset_provenance["c2pa_manifest"]
    display(Markdown("### C2PA manifest"))
    display_table(
        ["Field", "Value"],
        [
            ("status", c2pa["status"]),
            ("validation_state", c2pa["validation_state"]),
            ("signature_valid", c2pa["signature_valid"]),
            ("trusted", c2pa["trusted"]),
            ("issuer", c2pa["issuer"]),
            ("actions", len(c2pa["actions"])),
            ("AI classification", c2pa["ai_usage"]["classification"]),
            ("modified regions", len(c2pa["ai_usage"]["modified_regions"])),
            ("validation errors", len(c2pa["validation_errors"])),
            ("validation warnings", len(c2pa["validation_warnings"])),
        ],
    )

    software = asset_provenance["software"]
    display(Markdown("### Software evidence"))
    if software["candidates"]:
        display_table(
            ["Name", "Version", "Source"],
            [
                (item["name"], item["version"], item["source"])
                for item in software["candidates"]
            ],
        )
    else:
        display(Markdown("_No software identifier was found._"))

    project_file = asset_provenance["project_file"]
    display(Markdown("### Source-project candidates"))
    if project_file["candidates"]:
        display_table(
            ["Path", "Type", "Size", "Relationship", "Verified"],
            [
                (
                    item["path"],
                    item["extension"],
                    item["size_bytes"],
                    item["relationship_basis"],
                    item["relationship_verified"],
                )
                for item in project_file["candidates"]
            ],
        )
    else:
        display(Markdown("_No recognised source-project candidate was found._"))

    editing_history = asset_provenance["editing_history"]
    display(Markdown("### Editing history"))
    if editing_history["events"]:
        display_table(
            ["Time", "Action", "Record type", "Source", "Verified"],
            [
                (
                    event["time"],
                    event["action"],
                    event["record_type"],
                    event["source"],
                    event["verified"],
                )
                for event in editing_history["events"]
            ],
        )
    else:
        display(Markdown("_No editing-history event is available._"))

## 8. Save JSON outputs

In [ ]:
result_path, bundle_path = save_outputs(
    result,
    bundle,
    ROOT / "outputs",
)

print("Saved result:", result_path)
print("Saved evidence-chain bundle:", bundle_path)

## 9. Expected SampleDemo reference (rounded)

- Chain Coherence: **0.9801**
- Confidence: **0.9738**
- Evidence Coverage: **0.2500**
- Completeness: **0.6250**
- Storyboard → Animatic: **0.9670**
- Animatic → Final: **0.9870**
- Artefacts: **6**
- Relationships: **5**